# Depth Anything 3 (DA3) Usage Example

This notebook demonstrates how to use Depth Anything 3 for camera poses and depth estimation.


In [ ]:
import torch
from depth_anything_3.api import DepthAnything3
import os
import numpy as np
device = torch.device("cuda")

# Load Giant model (downloads from HF on first run)
model = DepthAnything3.from_pretrained("depth-anything/DA3-GIANT-1.1")
model = model.to(device=device)

# Point at your SceneNN images
import glob
images = sorted(glob.glob("/home/yash/gs-scenenn/data/scenenn/colmap/021_processed/images/*.png")) # Use first 10 images for testing

prediction = model.inference(
    images,
    infer_gs=True,
    export_dir="/home/yash/gs-scenenn/output/scenenn/DA3/021_60_DS_G11/",
    export_format="gs_ply"
)

print("Depth shape:", prediction.depth.shape)          # [N, H, W]
print("Extrinsics shape:", prediction.extrinsics.shape) # [N, 3, 4]
print("Intrinsics shape:", prediction.intrinsics.shape) # [N, 3, 3]     

[INFO ] using SwiGLU layer as FFN
[INFO ] Processed Images Done taking 0.1334226131439209 seconds. Shape:  torch.Size([60, 3, 378, 504])
[INFO ] Selecting reference view using strategy: saddle_balanced
[INFO ] Model Forward Pass Done. Time: 3.8902430534362793 seconds
[INFO ] Conversion to Prediction Done. Time: 0.03073287010192871 seconds


Saving video to /home/yash/gs-scenenn/output/scenenn/DA3/021_60_DS_G11/gs_video/0000_extend.mp4 24 ['-crf', '18', '-pix_fmt', 'yuv420p'] (378, 1016, 3) 268
Moviepy - Writing video /home/yash/gs-scenenn/output/scenenn/DA3/021_60_DS_G11/gs_video/0000_extend.mp4



Moviepy - Done !
[INFO ] Export Results Done. Time: 12.977781295776367 seconds
Depth shape: (60, 378, 504)
Extrinsics shape: (60, 3, 4)
Intrinsics shape: (60, 3, 3)


In [ ]:
import torch
from depth_anything_3.api import DepthAnything3
import os
import numpy as np
import glob
device = torch.device("cuda")

# Load Giant model (downloads from HF on first run)
model = DepthAnything3.from_pretrained("depth-anything/DA3NESTED-GIANT-LARGE-1.1")
model = model.to(device=device)

data_dir = "/home/yash/gs-scenenn/data/scenenn/colmap/021_batched/"
output_dir = "/home/yash/gs-scenenn/output/scenenn/DA3/021/"


batches = sorted(f for f in os.listdir(data_dir) if "batch" in f)

for batch in batches:
    batch_path = os.path.join(data_dir, batch)
    images = sorted(glob.glob(os.path.join(batch_path, "images/*.png")))
    
    prediction = model.inference(
        images,
        infer_gs=True,
        export_dir=os.path.join(output_dir, batch),
        export_format="gs_ply"
    )

    print(f"Processed {batch}: Depth shape: {prediction.depth.shape}, Extrinsics shape: {prediction.extrinsics.shape if prediction.extrinsics is not None else 'None'}, Intrinsics shape: {prediction.intrinsics.shape if prediction.intrinsics is not None else 'None'}")

In [ ]:
import torch
from depth_anything_3.api import DepthAnything3
import numpy as np, glob

device = torch.device("cuda")
model = DepthAnything3.from_pretrained("depth-anything/DA3-GIANT-1.1").to(device)

images = sorted(glob.glob("/home/yash/gs-scenenn/data/scenenn/colmap/005_batched/batch_00/images/*.png"))

ext_np = np.load("/home/yash/gs-scenenn/data/scenenn/colmap/005_batched/batch_00/extrinsics.npy")  # [N, 4, 4] c2w float32
ixt_np = np.load("/home/yash/gs-scenenn/data/scenenn/colmap/005_batched/batch_00/intrinsics.npy")  # [N, 3, 3] float32

# Convert c2w → w2c as numpy (DA3 expects w2c, handles batch dim itself)
c2w = torch.from_numpy(ext_np).float()
w2c_np = torch.linalg.inv(c2w).numpy()  # [N, 4, 4] w2c

prediction = model.inference(
    images,
    extrinsics=w2c_np,          # [N, 4, 4] numpy — DA3 adds batch dim internally
    intrinsics=ixt_np,          # [N, 3, 3] numpy
    infer_gs=True,
    export_dir="/home/yash/gs-scenenn/output/scenenn/DA3/005_60_OF_00_EI_G11",
    export_format="gs_ply"
)

[INFO ] using SwiGLU layer as FFN
[INFO ] Processed Images Done taking 0.1255323886871338 seconds. Shape:  torch.Size([60, 3, 378, 504])
[INFO ] Using camera conditions provided by the user
[INFO ] Model Forward Pass Done. Time: 3.808048963546753 seconds
[INFO ] Conversion to Prediction Done. Time: 0.02950119972229004 seconds


Saving video to /home/yash/gs-scenenn/output/scenenn/DA3/005_60_OF_00_EI_G11/gs_video/0000_extend.mp4 24 ['-crf', '18', '-pix_fmt', 'yuv420p'] (378, 1016, 3) 268
Moviepy - Writing video /home/yash/gs-scenenn/output/scenenn/DA3/005_60_OF_00_EI_G11/gs_video/0000_extend.mp4



Moviepy - Done !
[INFO ] Export Results Done. Time: 13.217338562011719 seconds


In [ ]:
# import numpy as np
# import open3d as o3d
# from plyfile import PlyData

# import torch
# import open3d as o3d
# import numpy as np

# # pred_pcd = o3d.io.read_point_cloud("/home/yash/Depth-Anything-3/notebooks/gs_60_imgs/gs_ply/0000.ply")
# gt_pcd   = o3d.io.read_point_cloud("/home/yash/gs-scenenn/data/scenenn/raw/005/005.ply")

# # === 1. Load Gaussian PLY (extract centers only) ===
# print("Loading DA3 Gaussians...")
# plydata = PlyData.read("/home/yash/Depth-Anything-3/notebooks/gs_90_imgs/gs_ply/0000.ply")
# vertex = plydata['vertex']
# xyz = np.vstack([vertex['x'], vertex['y'], vertex['z']]).T

# pred_pcd = o3d.geometry.PointCloud()
# pred_pcd.points = o3d.utility.Vector3dVector(xyz)

# # Load GT
# # gt_pcd = o3d.io.read_point_cloud("scenenn_016_gt.ply")
# print(f"Loaded: {len(pred_pcd.points)} pred, {len(gt_pcd.points)} GT points")

# # === 2. Align and Rescale ===
# print("\nAligning point clouds...")
# pred_extent = np.linalg.norm(pred_pcd.get_axis_aligned_bounding_box().get_extent())
# gt_extent = np.linalg.norm(gt_pcd.get_axis_aligned_bounding_box().get_extent())
# scale_factor = gt_extent / pred_extent

# pred_pcd.scale(scale_factor, center=pred_pcd.get_center())

# pred_center = pred_pcd.get_axis_aligned_bounding_box().get_center()
# gt_center = gt_pcd.get_axis_aligned_bounding_box().get_center()
# pred_pcd.translate(gt_center - pred_center)

# # ICP refinement
# result = o3d.pipelines.registration.registration_icp(
#     pred_pcd, gt_pcd,
#     max_correspondence_distance=0.05,
#     init=np.eye(4)
# )
# pred_pcd.transform(result.transformation)
# print(f"ICP fitness: {result.fitness:.4f}")

# # === 3. Downsample ===
# print("\nDownsampling...")
# voxel_size = 0.01  # 1cm voxels
# pred_pcd_down = pred_pcd.voxel_down_sample(voxel_size=voxel_size)
# gt_pcd_down = gt_pcd.voxel_down_sample(voxel_size=voxel_size)
# print(f"After downsampling: {len(pred_pcd_down.points)} pred, {len(gt_pcd_down.points)} GT")

# # === 4. Compute Metrics ===
# print("\nComputing metrics...")
# d1 = np.asarray(pred_pcd_down.compute_point_cloud_distance(gt_pcd_down))
# d2 = np.asarray(gt_pcd_down.compute_point_cloud_distance(pred_pcd_down))

# chamfer = (d1.mean() + d2.mean()) / 2

# thresh = 0.05  # 5cm
# precision = (d1 < thresh).mean()
# recall = (d2 < thresh).mean()
# f1 = 2 * precision * recall / (precision + recall + 1e-8)

# print("\n" + "="*60)
# print("RESULTS")
# print("="*60)
# print(f"Chamfer Distance: {chamfer:.6f} m")
# print(f"Precision@5cm: {precision:.4f}")
# print(f"Recall@5cm: {recall:.4f}")
# print(f"F1-score@5cm: {f1:.4f}")
# print(f"\nDistance stats (pred→GT):")
# print(f"  Median: {np.median(d1):.4f} m")
# print(f"  Mean:   {d1.mean():.4f} m")

In [4]:

# print("="*60)
# print("POINT CLOUD STATS")
# print("="*60)
# print(f"Pred points: {len(pred_pcd.points)}")
# print(f"GT points:   {len(gt_pcd.points)}")

# pred_extent = pred_pcd.get_axis_aligned_bounding_box().get_extent()
# gt_extent   = gt_pcd.get_axis_aligned_bounding_box().get_extent()
# print(f"\nPred extent: {pred_extent}")
# print(f"GT extent:   {gt_extent}")
# print(f"Scale ratio: {np.linalg.norm(pred_extent) / np.linalg.norm(gt_extent):.2f}x")

# pred_center = pred_pcd.get_axis_aligned_bounding_box().get_center()
# gt_center   = gt_pcd.get_axis_aligned_bounding_box().get_center()
# print(f"\nPred center: {pred_center}")
# print(f"GT center:   {gt_center}")
# print(f"Center distance: {np.linalg.norm(pred_center - gt_center):.4f}")

# # Distance distribution
# d1 = np.asarray(pred_pcd.compute_point_cloud_distance(gt_pcd))
# print("\n" + "="*60)
# print("DISTANCE STATS (pred → GT)")
# print("="*60)
# # print(f"Min:    {d1.min():.4f}")
# print(f"10th %: {np.percentile(d1, 10):.4f}")
# print(f"Median: {np.median(d1):.4f}")
# print(f"Mean:   {d1.mean():.4f}")
# print(f"90th %: {np.percentile(d1, 90):.4f}")
# print(f"Max:    {d1.max():.4f}")

# # Visualize
# pred_pcd.paint_uniform_color([1, 0, 0])
# gt_pcd.paint_uniform_color([0, 1, 0])
# o3d.visualization.draw_geometries([pred_pcd, gt_pcd])

In [ ]:
import os
import glob
import numpy as np
import torch
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import lpips

# Load LPIPS model
lpips_fn = lpips.LPIPS(net='alex').cuda()

# Paths to extracted frames
gt_frames = sorted(glob.glob("gt_frames/*.png"))
pred_frames = sorted(glob.glob("da3_rendered_frames/*.png"))

psnr_scores = []
ssim_scores = []
lpips_scores = []

for gt_path, pred_path in zip(gt_frames, pred_frames):
    # Load images
    gt_img = np.array(Image.open(gt_path).convert('RGB'))
    pred_img = np.array(Image.open(pred_path).convert('RGB'))
    
    # PSNR
    psnr_val = psnr(gt_img, pred_img, data_range=255)
    psnr_scores.append(psnr_val)
    
    # SSIM
    ssim_val = ssim(gt_img, pred_img, channel_axis=-1, data_range=255)
    ssim_scores.append(ssim_val)
    
    # LPIPS (needs [0,1] range, CHW format)
    gt_tensor = torch.from_numpy(gt_img).permute(2,0,1).float() / 255.0
    pred_tensor = torch.from_numpy(pred_img).permute(2,0,1).float() / 255.0
    gt_tensor = gt_tensor.unsqueeze(0).cuda()
    pred_tensor = pred_tensor.unsqueeze(0).cuda()
    
    lpips_val = lpips_fn(gt_tensor, pred_tensor).item()
    lpips_scores.append(lpips_val)

# Average across all frames
print(f"PSNR: {np.mean(psnr_scores):.2f}")
print(f"SSIM: {np.mean(ssim_scores):.4f}")
print(f"LPIPS: {np.mean(lpips_scores):.4f}")